In [1]:
!pip install rank_bm25

Defaulting to user installation because normal site-packages is not writeable


**Rank-BM25(Best Matching 25)** is essentially the "gold standard" algorithm for keyword-based search.

    Python: The rank_bm25 library is the most popular lightweight implementation.

    Elasticsearch / Lucene: BM25 is the default ranking algorithm used by these industry-standard search engines.

**Working**

    It is a widely used, highly effective ranking function that ranks documents based on their relevance to a search query. 

    It improves on TF-IDF by incorporating **term frequency saturation** (limiting the impact of high-frequency words) and **document length normalization** (preventing long documents from ranking higher simply due to word count)


In [1]:
#!pip install --upgrade pip

In [1]:
from langchain_aws import ChatBedrockConverse, BedrockEmbeddings
#llm = ChatBedrockConverse(model_id="amazon.nova-lite-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=50)

llm = ChatBedrockConverse(model_id="cohere.command-r-plus-v1:0", region_name="us-east-1", temperature=0.5, max_tokens=500)

embedding_model = BedrockEmbeddings(model_id="cohere.embed-english-v3")

In [4]:
import warnings as ws 
ws.filterwarnings("ignore")

In [5]:
from langchain_classic.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

# 1. Load and Split
loader = PyPDFLoader("research_paper_1.pdf")
docs = loader.load()
docs[0:2]

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Published as a conference paper at ICLR 2025\nAGENTIC AI FOR SCIENTIFIC DISCOVERY : A S URVEY\nOF PROGRESS , C HALLENGES , AND FUTURE DIREC -\nTIONS\nMourad Gridach, Jay Nanavati, Khaldoun Zine El Abidine, Lenon Mendes & Christina Mack\nIQVIA\n{firstname.lastname}@iqvia.com\nABSTRACT\nThe integration of Agentic AI into scientific discovery marks a new frontier in\nresearch automation. These AI systems, capable of reasoning, planning, and au-\ntonomous decision-making, are transforming how scientists perform literature re-\nview, generate hypotheses, conduct experiments, and analyze results. This sur-\nvey provides a comprehensive overview of Agentic AI for scientific discovery,\ncategorizing existing systems and tools, and highlighting recent progress across\nfields such as chemistry, biology,

In [6]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
splits[0:5]

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Published as a conference paper at ICLR 2025\nAGENTIC AI FOR SCIENTIFIC DISCOVERY : A S URVEY\nOF PROGRESS , C HALLENGES , AND FUTURE DIREC -\nTIONS\nMourad Gridach, Jay Nanavati, Khaldoun Zine El Abidine, Lenon Mendes & Christina Mack\nIQVIA\n{firstname.lastname}@iqvia.com\nABSTRACT\nThe integration of Agentic AI into scientific discovery marks a new frontier in\nresearch automation. These AI systems, capable of reasoning, planning, and au-\ntonomous decision-making, are transforming how scientists perform literature re-\nview, generate hypotheses, conduct experiments, and analyze results. This sur-\nvey provides a comprehensive overview of Agentic AI for scientific discovery,\ncategorizing existing systems and tools, and highlighting recent progress across\nfields such as chemistry, biology,

In [7]:
## 2. Vector Store
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)

In [8]:
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})


In [9]:
# 3. Keyword Search (BM25) - Great for specific names like "Ashwatthama"
bm25_retriever = BM25Retriever.from_documents(splits)
bm25_retriever.k = 2

In [10]:
# 4. Hybrid Combo
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever], 
    weights=[0.5, 0.5]  #weights: Determines the influence of each retriever.
)

In [11]:
print(bm25_retriever.invoke("What are agents and agentic ai?"))

[Document(metadata={'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20260117133359', 'source': 'research_paper_1.pdf', 'total_pages': 6, 'page': 5, 'page_label': '6'}, page_content='farollahi & Buehler (2024b); Schmidgall et al. (2025) as well as machine learning Li et al. (2024b);\nHuang et al. (2024b); Chan et al. among others. Figure 2 shows the summary of these agentic AI\nframeworks for scientific discovery in various domains.\nFigure 1: Agentic AI workflow for scien-\ntific discovery.\nFigure 2: AI Agents frameworks for scientific dis-\ncovery.\n6 I MPLEMENTATION TOOLS , DATASETS AND METRICS\nThe development and evaluation of agentic AI systems for scientific discovery rely on a robust tools,\ncurated datasets, and well-defined evaluation metrics. This section provides an overview of the key\nresources used in the field to facilitate the design, training, and assessment of autonomous AI agents\nfor scientific discovery.\n6'), Document(metadata={'producer': 'PDFium',

In [27]:
print(vector_retriever.invoke("What are agents and agentic ai?"))

[
    Document(
        metadata={
            'source': 'research_paper_1.pdf',
            'producer': 'PDFium',
            'creationdate': 'D:20260117133359',
            'page_label': '2',
            'total_pages': 6,
            'page': 1,
            'creator': 'PDFium'
        },
        page_content='Published as a conference paper at ICLR 2025\n2 A GENTIC AI: F OUNDATIONS AND KEY 
CONCEPTS\n2.1 D EFINITION AND CHARACTERISTICS\nThe concept of an “agent” has a rich history and has been explored 
across various disciplines in-\ncluding philosophy. It has been discussed by influential philosophers starting from
Aristotle to\nHume among others. Generally, an “agent” is an entity that has the ability to act, while the 
con-\ncept of “agency” refers to the exercise or representation of this ability Schlosser (2019). In 
artificial\nintelligence, an agent is an autonomous intelligent entity capable of performing appropriate 
and\ncontextually relevant actions in response to sensory input, whether operating in physical, virtual, 
or\nmixed-reality environments. Agentic AI introduces a new paradigm in the AI community, highlight-\ning the 
concept of embodied intelligence and showing the importance of an integrated framework'
    ),
    Document(
        metadata={
            'creationdate': 'D:20260117133359',
            'page_label': '2',
            'source': 'research_paper_1.pdf',
            'page': 1,
            'creator': 'PDFium',
            'total_pages': 6,
            'producer': 'PDFium'
        },
        page_content='mixed-reality environments. Agentic AI introduces a new paradigm in the AI community, 
highlight-\ning the concept of embodied intelligence and showing the importance of an integrated framework\nfor 
interactive agents within complex systems Huang et al. (2024c). This paradigm stems from the\nunderstanding that 
intelligence emerges from the intricate interaction between key processes such\nas autonomy, learning, memory, 
perception, planning, decision-making and action.\n2.2 S INGLE AGENT VS . M ULTI -AGENTS\nWith the explosion of 
both research papers and industrial applications of agentic AI, a new debate\nemerged on whether single or 
multi-agent systems are best suited for solving complex tasks. In\ngeneral, single agent architectures shine when 
dealing with well-defined problems and feedback\nfrom the user is not needed, while multi-agent architectures are 
suitable for solving problems that\ninvolve collaboration and multiple runs are needed.'
    )
]

In [34]:
context = hybrid_retriever.invoke("What are agents and agentic ai?")
print(context)

[
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 1,
            'page_label': '2'
        },
        page_content='mixed-reality environments. Agentic AI introduces a new paradigm in the AI community, 
highlight-\ning the concept of embodied intelligence and showing the importance of an integrated framework\nfor 
interactive agents within complex systems Huang et al. (2024c). This paradigm stems from the\nunderstanding that 
intelligence emerges from the intricate interaction between key processes such\nas autonomy, learning, memory, 
perception, planning, decision-making and action.\n2.2 S INGLE AGENT VS . M ULTI -AGENTS\nWith the explosion of 
both research papers and industrial applications of agentic AI, a new debate\nemerged on whether single or 
multi-agent systems are best suited for solving complex tasks. In\ngeneral, single agent architectures shine when 
dealing with well-defined problems and feedback\nfrom the user is not needed, while multi-agent architectures are 
suitable for solving problems that\ninvolve collaboration and multiple runs are needed.'
    ),
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 5,
            'page_label': '6'
        },
        page_content='farollahi & Buehler (2024b); Schmidgall et al. (2025) as well as machine learning Li et al. 
(2024b);\nHuang et al. (2024b); Chan et al. among others. Figure 2 shows the summary of these agentic 
AI\nframeworks for scientific discovery in various domains.\nFigure 1: Agentic AI workflow for scien-\ntific 
discovery.\nFigure 2: AI Agents frameworks for scientific dis-\ncovery.\n6 I MPLEMENTATION TOOLS , DATASETS AND 
METRICS\nThe development and evaluation of agentic AI systems for scientific discovery rely on a robust 
tools,\ncurated datasets, and well-defined evaluation metrics. This section provides an overview of the 
key\nresources used in the field to facilitate the design, training, and assessment of autonomous AI agents\nfor 
scientific discovery.\n6'
    ),
    Document(
        metadata={
            'total_pages': 6,
            'page_label': '2',
            'producer': 'PDFium',
            'creationdate': 'D:20260117133359',
            'creator': 'PDFium',
            'page': 1,
            'source': 'research_paper_1.pdf'
        },
        page_content='Published as a conference paper at ICLR 2025\n2 A GENTIC AI: F OUNDATIONS AND KEY 
CONCEPTS\n2.1 D EFINITION AND CHARACTERISTICS\nThe concept of an “agent” has a rich history and has been explored 
across various disciplines in-\ncluding philosophy. It has been discussed by influential philosophers starting from
Aristotle to\nHume among others. Generally, an “agent” is an entity that has the ability to act, while the 
con-\ncept of “agency” refers to the exercise or representation of this ability Schlosser (2019). In 
artificial\nintelligence, an agent is an autonomous intelligent entity capable of performing appropriate 
and\ncontextually relevant actions in response to sensory input, whether operating in physical, virtual, 
or\nmixed-reality environments. Agentic AI introduces a new paradigm in the AI community, highlight-\ning the 
concept of embodied intelligence and showing the importance of an integrated framework'
    )
]

In [35]:
from langchain_core.output_parsers import StrOutputParser

In [51]:
prompt = ChatPromptTemplate.from_template( """
        Respond to the user query based on the given context.
        If the context is not relevant to the query say context is not suffiecient to 
        answer this query and do not answer the query. 
        Query: {query}, context:{context}"""
    )

In [52]:
chain = prompt | llm | StrOutputParser()

## VectorDB Alone

In [55]:
while True:
    query=input("query:")
    if query=="exit":
        break
    print("retrieved context:",vector_retriever.invoke(query))
    print(chain.invoke({"query":query,
              "context":vector_retriever.invoke(query)}))

query: What are fully autonomous systems?


retrieved context:
[
    Document(
        metadata={
            'creationdate': 'D:20260117133359',
            'page': 2,
            'creator': 'PDFium',
            'page_label': '3',
            'producer': 'PDFium',
            'total_pages': 6,
            'source': 'research_paper_1.pdf'
        },
        page_content='Published as a conference paper at ICLR 2025\nproblem-solving. As the field continues to 
evolve, its applications in scientific discovery are ex-\npanding across diverse domains, from chemistry and 
biology to materials science and healthcare.\nAgentic AI systems can be broadly categorized based on their level of
autonomy, interaction with\nresearchers, and scope of application.\n3.1 F ULLY AUTONOMOUS SYSTEMS\nFully autonomous
systems are designed to operate independently, automating end-to-end scientific\nworkflows with minimal human 
intervention. These systems leverage advanced AI capabilities,\nsuch as natural language understanding, planning, 
and decision-making, to perform complex tasks\nranging from hypothesis generation to experiment execution. Boiko et
al. (2023) developed Co-\nscientist, an autonomous AI agent powered by GPT-4 that plans, designs, and executes 
chemical\nexperiments. Similarly, M. Bran et al. (2024) introduced ChemCrow, which extends the capabilities'
    ),
    Document(
        metadata={
            'page_label': '2',
            'producer': 'PDFium',
            'source': 'research_paper_1.pdf',
            'page': 1,
            'creator': 'PDFium',
            'total_pages': 6,
            'creationdate': 'D:20260117133359'
        },
        page_content='from the user is not needed, while multi-agent architectures are suitable for solving 
problems that\ninvolve collaboration and multiple runs are needed.\nSingle AgentIn nutshell, a single agent is able
to achieve its goal independently without relying on\nassistance or feedback from other AI agents, even if multiple
agents coexist within the same envi-\nronment. However, there may be opportunities for humans to be in the loop by 
providing feedback\nfor agent guidance. More specifically, a single agent with an LLM backbone capable of 
handling\nmultiple tasks and domains is called LM-based agent. It is able to perform reasoning, planning and\ntool 
execution on their own. Given an input prompt, an agent uses the tool to execute its task. Com-\nmon applications 
using a single agent include Scientific Discovery Lu et al. (2024); Ghafarollahi\n& Buehler (2024a); Kang & Xiong 
(2024); Xin et al. (2024), web scenarios Nakano et al. (2021);'
    )
]

Fully autonomous systems are artificial intelligence (AI) systems designed to operate independently, automating 
complex tasks with minimal human intervention. These systems utilize advanced AI capabilities, such as natural 
language understanding, planning, and decision-making, to manage end-to-end scientific workflows. An example of a 
fully autonomous system is an AI agent that can plan, design, and execute chemical experiments, showcasing its 
ability to operate autonomously in a specific domain.

query: exit


## bm25_retriever

In [57]:
while True:
    query=input("query:")
    if query=="exit":
        break
    print("retrieved context:",bm25_retriever.invoke(query))
    print(chain.invoke({"query":query,
              "context":bm25_retriever.invoke(query)}))

query: What are fully autonomous systems?


retrieved context:
[
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 2,
            'page_label': '3'
        },
        page_content='scientist, an autonomous AI agent powered by GPT-4 that plans, designs, and executes 
chemical\nexperiments. Similarly, M. Bran et al. (2024) introduced ChemCrow, which extends the capabilities\nof 
GPT-4 by integrating 18 expert-designed tools for tasks such as organic synthesis, drug discovery,\nand materials 
design. It demonstrates the potential of fully autonomous systems to tackle complex,\ndomain-specific challenges. 
ProtAgents Ghafarollahi & Buehler (2024a) was proposed for protein\ndesign and molecular modeling. It leverages 
LLMs and reinforcement learning to optimize protein\nstructures, predict folding patterns, and perform molecular 
docking simulations. ProtAgents can\nautonomously generate, test, and refine protein sequences to meet desired 
biochemical properties.\nLLaMP (Large Language Model for Materials Prediction) Chiang et al. (2024) is an 
autonomous\nAI agent for materials science, using RAG to predict material properties and optimize formula-'
    ),
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 3,
            'page_label': '4'
        },
        page_content='fields.\nDespite the progress made by existing frameworks, several challenges remain in 
automating the\nliterature review process. While frameworks such as SciLitLLM and ResearchArena 
demonstrate\npromising results, they often struggle with tasks requiring deep domain-specific knowledge and 
nu-\nanced understanding. This limitation is further highlighted in Agent Laboratory Schmidgall et al.\n(2025), 
where a significant performance drop was observed during the literature review phase, em-\nphasizing the complexity
of automating this process. Another challenge lies in human-AI collabora-\ntion, as many existing frameworks 
prioritize fully autonomous workflows. This approach may limit\nusability for researchers who want to explore their
unique ideas, underscoring the need for collabo-\nrative approaches that effectively integrate human expertise with
AI capabilities. Generalizability is\nalso a major obstacle, as many frameworks are designed for specific domains 
like machine learning,'
    )
]

Fully autonomous systems refer to AI agents or frameworks that can operate independently, without human 
intervention, to perform complex, domain-specific tasks. In the context of the provided text, examples of fully 
autonomous systems include "ChemCrow," which uses GPT-4 for tasks in organic synthesis and drug discovery, and 
"ProtAgents," which leverages LLMs and reinforcement learning for protein design and molecular modeling. These 
systems showcase the potential for AI to handle intricate and specialized challenges with minimal human input.

query: exit


In [58]:
while True:
    query=input("query:")
    if query=="exit":
        break
    print("retrieved context:",hybrid_retriever.invoke(query))
    print(chain.invoke({"query":query,
              "context":hybrid_retriever.invoke(query)}))

query: What are fully autonomous systems?


retrieved context:
[
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 2,
            'page_label': '3'
        },
        page_content='scientist, an autonomous AI agent powered by GPT-4 that plans, designs, and executes 
chemical\nexperiments. Similarly, M. Bran et al. (2024) introduced ChemCrow, which extends the capabilities\nof 
GPT-4 by integrating 18 expert-designed tools for tasks such as organic synthesis, drug discovery,\nand materials 
design. It demonstrates the potential of fully autonomous systems to tackle complex,\ndomain-specific challenges. 
ProtAgents Ghafarollahi & Buehler (2024a) was proposed for protein\ndesign and molecular modeling. It leverages 
LLMs and reinforcement learning to optimize protein\nstructures, predict folding patterns, and perform molecular 
docking simulations. ProtAgents can\nautonomously generate, test, and refine protein sequences to meet desired 
biochemical properties.\nLLaMP (Large Language Model for Materials Prediction) Chiang et al. (2024) is an 
autonomous\nAI agent for materials science, using RAG to predict material properties and optimize formula-'
    ),
    Document(
        metadata={
            'producer': 'PDFium',
            'total_pages': 6,
            'page_label': '3',
            'page': 2,
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf'
        },
        page_content='Published as a conference paper at ICLR 2025\nproblem-solving. As the field continues to 
evolve, its applications in scientific discovery are ex-\npanding across diverse domains, from chemistry and 
biology to materials science and healthcare.\nAgentic AI systems can be broadly categorized based on their level of
autonomy, interaction with\nresearchers, and scope of application.\n3.1 F ULLY AUTONOMOUS SYSTEMS\nFully autonomous
systems are designed to operate independently, automating end-to-end scientific\nworkflows with minimal human 
intervention. These systems leverage advanced AI capabilities,\nsuch as natural language understanding, planning, 
and decision-making, to perform complex tasks\nranging from hypothesis generation to experiment execution. Boiko et
al. (2023) developed Co-\nscientist, an autonomous AI agent powered by GPT-4 that plans, designs, and executes 
chemical\nexperiments. Similarly, M. Bran et al. (2024) introduced ChemCrow, which extends the capabilities'
    ),
    Document(
        metadata={
            'producer': 'PDFium',
            'creator': 'PDFium',
            'creationdate': 'D:20260117133359',
            'source': 'research_paper_1.pdf',
            'total_pages': 6,
            'page': 3,
            'page_label': '4'
        },
        page_content='fields.\nDespite the progress made by existing frameworks, several challenges remain in 
automating the\nliterature review process. While frameworks such as SciLitLLM and ResearchArena 
demonstrate\npromising results, they often struggle with tasks requiring deep domain-specific knowledge and 
nu-\nanced understanding. This limitation is further highlighted in Agent Laboratory Schmidgall et al.\n(2025), 
where a significant performance drop was observed during the literature review phase, em-\nphasizing the complexity
of automating this process. Another challenge lies in human-AI collabora-\ntion, as many existing frameworks 
prioritize fully autonomous workflows. This approach may limit\nusability for researchers who want to explore their
unique ideas, underscoring the need for collabo-\nrative approaches that effectively integrate human expertise with
AI capabilities. Generalizability is\nalso a major obstacle, as many frameworks are designed for specific domains 
like machine learning,'
    ),
    Document(
        metadata={
            

Fully autonomous systems are designed to operate independently, executing end-to-end scientific workflows with 
minimal human intervention. These systems utilize advanced AI capabilities, including natural language 
understanding, planning, and decision-making, to perform a range of complex tasks. The goal of fully autonomous 
systems is to automate scientific processes, from generating hypotheses to conducting experiments, without 
requiring significant input from human researchers. Examples of fully autonomous systems include Co-scientist, an 
AI agent powered by GPT-4 that plans and executes chemical experiments, and ChemCrow, which builds upon GPT-4 to 
integrate expert-designed tools for organic synthesis, drug discovery, and materials design.

query: exit
